# 10 - Bonus: Deep Learning with PyTorch

Optional deep learning examples, for those who want the same classification problem in
torch rather than scikit-learn.

**PyTorch is not installed by default** — it is a ~200 MB download and this is the last
of twenty-six notebooks. If the cell below says it is not installed, that is expected and
nothing is broken; run `make bonus` and re-run this notebook to see the model. Everything
in workshops 2 and 3 works without it.

In [1]:
import sys
from pathlib import Path

# workshop_3/ holds _fetch.py and _sources.py; src/ holds the project's own helpers.
# When opened in Jupyter Lab with Workshop/ as root, Path.cwd() is Workshop/
# so we need to add workshop_3/ and src/ to the path.
for p in (Path.cwd() / "workshop_3", Path.cwd() / "src"):
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p.resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import _fetch
import _sources as S
from ocean_data_workshop.dsn import dsn
import psycopg

# ML imports
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split

# Setup
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

print("ML Workshop Environment Ready")
print("=" * 60)
print("Data sources:")
print("  - NDBC 46092 (wind)")
print("  - GLORYS12V1 (ocean)")
print("  - SanctSound MB01 (acoustic)")
print("=" * 60)

ML Workshop Environment Ready
Data sources:
  - NDBC 46092 (wind)
  - GLORYS12V1 (ocean)
  - SanctSound MB01 (acoustic)


In [2]:
# PyTorch is optional -- it is a large download and notebook 09 is the last one, so
# nobody should be waiting on it. Notebooks 02 and 07 do the same three problems with
# scikit-learn in seconds; this one is here for the people who want to see the
# equivalent in torch.
#
# The ImportError is scoped to the import alone, on purpose. Wrapping the whole cell in
# try/except ImportError looks harmless and is not: it swallows every error raised after
# the import as if torch were merely absent, so a typo in the model below reports itself
# as "PyTorch not installed" and this notebook stayed green for months while broken.
try:
    import torch
    import torch.nn as nn
except ImportError:
    print("PyTorch is not installed -- that is fine, this notebook is optional.")
    print("  install it with:  make bonus")
    print("  or:               uv sync --extra bonus")
    print()
    print("Everything in workshops 2 and 3 runs without it. Notebooks 02 (classification)")
    print("and 07 (clustering) cover the same ground with scikit-learn.")
else:
    print(f"PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")

    # The same three-layer stack, in torch. 30 input bands is what the acoustic features
    # are, and 30 -> 64 -> 32 -> 1 is the shape notebook 02 uses in sklearn.
    n_bands = 30

    class DolphinNet(nn.Module):
        def __init__(self, input_size: int = n_bands):
            super().__init__()
            self.fc1 = nn.Linear(input_size, 64)
            self.fc2 = nn.Linear(64, 32)
            self.fc3 = nn.Linear(32, 1)
            self.relu = nn.ReLU()
            self.dropout = nn.Dropout(0.3)

        def forward(self, x):
            x = self.dropout(self.relu(self.fc1(x)))
            x = self.dropout(self.relu(self.fc2(x)))
            return torch.sigmoid(self.fc3(x))

    net = DolphinNet()
    n_params = sum(p.numel() for p in net.parameters())
    print(f"  DolphinNet: {n_bands} -> 64 -> 32 -> 1, {n_params:,} parameters")
    print(f"  forward pass on a batch of 4: {tuple(net(torch.zeros(4, n_bands)).shape)}")

PyTorch is not installed -- that is fine, this notebook is optional.
  install it with:  make bonus
  or:               uv sync --extra bonus

Everything in workshops 2 and 3 runs without it. Notebooks 02 (classification)
and 07 (clustering) cover the same ground with scikit-learn.
